# Notebook 04 — Pipeline complet QUBO-Assignation → QMKL → Classification

**Objectif :** valider l'hypothèse principale H1 sur 2 datasets :

> *"L'assignation QUBO produit un QMKL significativement meilleur que les baselines naïves"*

## Pipeline
```
Données → QuantumScaler → alignments marginaux a[k,m]
       → QUBO matrix → SA solver → assignation optimale
       → M kernels analytiques → QMKL (centered alignment)
       → SVM precomputed → AUC (5-fold, 20 répétitions)
```

## Méthodes comparées
1. Aléatoire (mean sur 20 seeds)
2. Non-overlapping fixe
3. **QUBO-assignation** (SA solver)
4. RBF-SVM (classique)
5. Random Forest (classique)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon

from src.data.loaders import load_breast_cancer_data, load_german_credit, subsample
from src.preprocessing.scaler import QuantumScaler
from src.kernels.analytical import KERNEL_CONFIGS
from src.kernels.subset_kernels import (
    non_overlapping_subsets, random_subsets,
    build_subset_kernels_train_test
)
from src.qubo.assignment_qubo import (
    compute_marginal_alignments, build_qubo_matrix, decode_assignment
)
from src.qubo.solvers import solve_simulated_annealing
from src.mkl.combiner import MultipleKernelCombiner

plt.rcParams.update({'font.family': 'sans-serif', 'figure.dpi': 130, 'savefig.bbox': 'tight'})
print('Setup OK')

In [ ]:
# Paramètres
Q = 4
M = 5
N_SAMPLES = 150
N_REPS    = 20
N_FOLDS   = 5

kernel_configs = [
    ('Z',  0.5), ('ZZ', 1.0), ('XZ', 0.5), ('ZZ', 2.0), ('Z', 2.0)
]

datasets = {
    'Breast Cancer': (load_breast_cancer_data, {}),
    'German Credit':  (load_german_credit, {}),
}

def cv_auc(X, y, assignment, kc, n_splits=5, C=1.0, seed=0):
    """AUC 5-fold avec QMKL centered alignment."""
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    aucs = []
    for tr, te in kf.split(X, y):
        K_trs, K_tes = build_subset_kernels_train_test(X[tr], X[te], assignment, kc)
        cb = MultipleKernelCombiner(method='centered')
        svm = SVC(kernel='precomputed', C=C, probability=True)
        svm.fit(cb.fit_combine(K_trs, y[tr]), y[tr])
        aucs.append(roc_auc_score(y[te], svm.predict_proba(cb.combine(K_tes))[:, 1]))
    return aucs

print('Fonctions prêtes')

## Expériences sur chaque dataset

In [ ]:
all_results = {}

for ds_name, (loader_fn, loader_kwargs) in datasets.items():
    print(f'\n{'='*55}')
    print(f'  Dataset : {ds_name}')
    print(f'{'='*55}')

    X_raw, y_raw, _ = loader_fn(**loader_kwargs)
    X_raw, y_raw = subsample(X_raw, y_raw, N_SAMPLES, seed=42)
    X = QuantumScaler().fit_transform(X_raw)
    d = X.shape[1]

    # ── Baseline aléatoire (20 seeds) ─────────────────────────────────
    print(f'  [1/5] Aléatoire ({N_REPS} seeds)...')
    rand_aucs = []
    for seed in range(N_REPS):
        asgn = random_subsets(d, Q, M, seed=seed)
        rand_aucs.extend(cv_auc(X, y_raw, asgn, kernel_configs, seed=seed))

    # ── Non-overlapping ────────────────────────────────────────────────
    print(f'  [2/5] Non-overlapping...')
    asgn_nolap = non_overlapping_subsets(d, Q, M)
    nolap_aucs = cv_auc(X, y_raw, asgn_nolap, kernel_configs, seed=42)

    # ── QUBO-assignation (SA) ──────────────────────────────────────────
    print(f'  [3/5] QUBO-assignation (SA)...')
    a = compute_marginal_alignments(X, y_raw, M, Q, kernel_configs, padding='top')
    Q_mat = build_qubo_matrix(a, d, M, Q, lambda_div=0.5, mu1=2.0)
    res = solve_simulated_annealing(Q_mat, d, M, Q, n_iter=50_000, seed=42)
    asgn_qubo = decode_assignment(res['x'], d, M, Q)
    qubo_aucs = []
    for rep in range(N_REPS):
        qubo_aucs.extend(cv_auc(X, y_raw, asgn_qubo, kernel_configs, seed=rep))

    # ── RBF-SVM ────────────────────────────────────────────────────────
    print(f'  [4/5] RBF-SVM...')
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    rbf_aucs = []
    for tr, te in kf.split(X, y_raw):
        clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
        clf.fit(X[tr], y_raw[tr])
        rbf_aucs.append(roc_auc_score(y_raw[te], clf.predict_proba(X[te])[:, 1]))

    # ── Random Forest ──────────────────────────────────────────────────
    print(f'  [5/5] Random Forest...')
    rf_aucs = []
    for tr, te in kf.split(X, y_raw):
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(X[tr], y_raw[tr])
        rf_aucs.append(roc_auc_score(y_raw[te], clf.predict_proba(X[te])[:, 1]))

    all_results[ds_name] = {
        'Random':  np.array(rand_aucs),
        'Non-overlap': np.array(nolap_aucs),
        'QUBO-assign': np.array(qubo_aucs),
        'RBF-SVM': np.array(rbf_aucs),
        'Rand Forest': np.array(rf_aucs),
    }

    print(f'\n  Résultats {ds_name}:')
    for name, aucs in all_results[ds_name].items():
        print(f'    {name:15s}: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}')

print('\n✓ Expériences terminées')

## Test statistique de Wilcoxon

H₀ : QUBO-assignation = Aléatoire  
H₁ : QUBO-assignation > Aléatoire

In [ ]:
for ds_name, res in all_results.items():
    qubo = res['QUBO-assign']
    rand = res['Random']
    n = min(len(qubo), len(rand))
    stat, pval = wilcoxon(qubo[:n], rand[:n], alternative='greater')
    print(f'{ds_name}:')
    print(f'  QUBO mean={np.mean(qubo):.3f}  Random mean={np.mean(rand):.3f}')
    print(f'  Wilcoxon p-value={pval:.4f}  {'*** SIGNIFICATIF' if pval < 0.05 else '(non significatif)'}')
    print()

## Visualisation finale

In [ ]:
fig, axes = plt.subplots(1, len(all_results), figsize=(7 * len(all_results), 5))
if len(all_results) == 1:
    axes = [axes]

colors_map = {
    'Random': '#95a5a6',
    'Non-overlap': '#3498db',
    'QUBO-assign': '#e74c3c',
    'RBF-SVM': '#2ecc71',
    'Rand Forest': '#f39c12',
}

for ax, (ds_name, res) in zip(axes, all_results.items()):
    methods = list(res.keys())
    means = [np.mean(res[m]) for m in methods]
    stds  = [np.std(res[m])  for m in methods]
    colors_list = [colors_map[m] for m in methods]

    bars = ax.bar(methods, means, yerr=stds, capsize=5,
                   color=colors_list, alpha=0.85, width=0.6)
    ax.set_ylim(max(0, min(means) - 0.05), min(1.0, max(means) + 0.05))
    ax.set_ylabel('AUC')
    ax.set_title(f'{ds_name}\n(N={N_SAMPLES}, Q={Q}, M={M})')
    ax.tick_params(axis='x', rotation=25)

    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + s + 0.003,
                f'{m:.3f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('QUBO-Assignation vs Baselines — AUC (5-fold CV)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/04_main_results.png')
plt.show()

## Visualisation de l'assignation optimale QUBO

In [ ]:
# Heatmap de l'assignation (feature x kernel)
ds_name = list(all_results.keys())[0]
X_raw, y_raw, feat_names = list(datasets.values())[0][0]()
X_raw, y_raw = subsample(X_raw, y_raw, N_SAMPLES, seed=42)
X = QuantumScaler().fit_transform(X_raw)
d = X.shape[1]

a = compute_marginal_alignments(X, y_raw, M, Q, kernel_configs, padding='top')
Q_mat_plot = build_qubo_matrix(a, d, M, Q, lambda_div=0.5, mu1=2.0)
res_plot = solve_simulated_annealing(Q_mat_plot, d, M, Q, n_iter=50_000, seed=42)
asgn_plot = decode_assignment(res_plot['x'], d, M, Q)

# Build assignment matrix
X_assign = np.zeros((d, M))
for m_id, feats in asgn_plot.items():
    for k in feats:
        X_assign[k, m_id] = 1

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(X_assign, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set_xlabel('Kernel m')
ax.set_ylabel('Feature k')
ax.set_title(f'Assignation QUBO optimale — {ds_name}\n(rouge = feature assignée à ce kernel)')
ax.set_xticks(range(M))
ax.set_xticklabels([f'K{m}\n({kernel_configs[m][0]},α={kernel_configs[m][1]})' for m in range(M)])
if len(feat_names) == d:
    ax.set_yticks(range(d))
    ax.set_yticklabels(feat_names, fontsize=7)
plt.colorbar(im, ax=ax, label='Assigné (1=oui)')
plt.tight_layout()
plt.savefig('../results/figures/04_qubo_assignment_heatmap.png')
plt.show()

## Conclusion

**H1 validée** si p-value Wilcoxon < 0.05 sur au moins 1 dataset.

**Interprétation** : l'assignation QUBO identifie les sous-ensembles de features
les plus informatifs pour chaque famille de kernel, sans PCA globale.

**Suite → Notebook 05 : test sur IBM Torino**